In [131]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [132]:
%pip install --quiet --upgrade langchain-text-splitters langchain-community langgraph

In [133]:
# use LangSmith to track traces (optional)

# import getpass
import os

os.environ["LANGSMITH_TRACING"] = "false" #set true if you want to use LangSmith
# os.environ["LANGSMITH_API_KEY"] = getpass.getpass()

## Using VertexAIEmbeddings, Google Gemini, Chroma(Vector DB)

In [134]:
%pip install -qU "langchain[google-vertexai]"

In [135]:
import vertexai
vertexai.init(project='cs391-project')

In [136]:
# Ensure your VertexAI credentials are configured
import os

KEYFILE_PATH = '/content/drive/MyDrive/cs391-project-11f0f788cfea.json'
os.environ["GOOGLE_APPLICATION_CREDENTIALS"] = KEYFILE_PATH

In [137]:
from langchain.chat_models import init_chat_model

llm = init_chat_model("gemini-2.0-flash-001", model_provider="google_vertexai")

In [138]:
from langchain_google_vertexai import VertexAIEmbeddings
from google.oauth2 import service_account

# Create credentials object
credentials = service_account.Credentials.from_service_account_file(KEYFILE_PATH)

# Pass credentials to the embeddings model
embeddings = VertexAIEmbeddings(
    model="text-embedding-004",
    credentials=credentials
)

In [139]:
%pip install -qU langchain-chroma

In [140]:
from langchain_chroma import Chroma

vector_store = Chroma(
    collection_name="example_collection",
    embedding_function=embeddings,
    persist_directory="/content/drive/MyDrive/chroma_langchain_db",  # Where to save data locally, remove if not necessary
)

## Basic Retrieval Process

In [141]:
import bs4
from langchain import hub
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langgraph.graph import START, StateGraph
from typing_extensions import List, TypedDict
from langchain_community.document_loaders import TextLoader

loader = TextLoader("/content/drive/MyDrive/RAG_data/merged_transcript.txt")

docs = loader.load()

text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
all_splits = text_splitter.split_documents(docs)

# Index chunks
_ = vector_store.add_documents(documents=all_splits)

# Define prompt for question-answering
prompt = hub.pull("rlm/rag-prompt")


# Define state for application
class State(TypedDict):
    question: str
    context: List[Document]
    answer: str


# Define application steps
def retrieve(state: State):
    retrieved_docs = vector_store.similarity_search(state["question"])
    return {"context": retrieved_docs}


def generate(state: State):
    docs_content = "\n\n".join(doc.page_content for doc in state["context"])
    messages = prompt.invoke({"question": state["question"], "context": docs_content})
    response = llm.invoke(messages)
    return {"answer": response.content}

# Compile application and test
graph_builder = StateGraph(State).add_sequence([retrieve, generate])
graph_builder.add_edge(START, "retrieve")
graph = graph_builder.compile()

/usr/local/lib/python3.11/dist-packages/langsmith/client.py:280: LangSmithMissingAPIKeyWarning: API key must be provided when using hosted LangSmith API
  warnings.warn(


In [142]:
response = graph.invoke({"question": "What is Task Decomposition?"})
print(response["answer"])

Task decomposition involves partitioning data into $k$ subsets. The specific nature of the data and the criteria for partitioning aren't provided in the context. Therefore, further information is needed for a complete definition.



In [143]:
response = graph.invoke({"question": "What are Support Vector Machines?"})
print(response["answer"])

Support Vector Machines were popular methods of doing classification before deep learning became widespread. They are related to logistic regression. Understanding them is valuable even with current deep learning methodologies.



## Using additional context data (class notes in Latex)

In [144]:
import os
import re

def merge_latex_files(input_files, output_file, prefix_text="Note"):
    """
    Merge multiple LaTeX files into one, adding markers to indicate source files.

    Args:
        input_files (list): List of input LaTeX file paths
        output_file (str): Output LaTeX file path
        prefix_text (str): Text to use as prefix for source indicators (default: "Note")
    """
    with open(output_file, 'w', encoding='utf-8') as outfile:
        # Write preamble - we'll take it from the first file
        if input_files:
            with open(input_files[0], 'r', encoding='utf-8') as first_file:
                content = first_file.read()

                # Extract the preamble (everything before \begin{document})
                preamble_match = re.search(r'(.*?)\\begin{document}', content, re.DOTALL)
                if preamble_match:
                    preamble = preamble_match.group(1)
                    outfile.write(preamble + "\n")
                else:
                    # If no \begin{document} found, assume it's a fragment and write a default preamble
                    outfile.write("\\documentclass{article}\n")
                    outfile.write("\\usepackage{amsmath}\n")
                    outfile.write("\\usepackage{amssymb}\n")
                    outfile.write("\\usepackage{graphicx}\n")
                    outfile.write("\\usepackage{hyperref}\n\n")

        # Write document start
        outfile.write("\\begin{document}\n\n")

        # Process each file
        for i, file_path in enumerate(input_files, 1):
            # Write a section header to indicate the source file
            file_name = os.path.basename(file_path)
            outfile.write(f"\\section*{{{prefix_text} {i}: {file_name}}}\n")
            outfile.write(f"\\addcontentsline{{toc}}{{section}}{{{prefix_text} {i}: {file_name}}}\n\n")

            with open(file_path, 'r', encoding='utf-8') as infile:
                content = infile.read()

                # Extract just the document body
                body_match = re.search(r'\\begin{document}(.*?)\\end{document}', content, re.DOTALL)
                if body_match:
                    # If the file has document tags, extract just the content between them
                    body = body_match.group(1)
                    outfile.write(body + "\n\n")
                else:
                    # If no document tags, assume the whole file is content
                    outfile.write(content + "\n\n")

        # Write document end
        outfile.write("\\end{document}\n")

In [145]:
# Check if Latex files are stored in the right directory

OUTPUT_FILE_PATH = '/content/drive/MyDrive/RAG_data/note_merged.tex'
latex_file_list = [f'/content/drive/MyDrive/RAG_data/note_{num}.tex' for num in range (1, 14)]
merge_latex_files(latex_file_list, OUTPUT_FILE_PATH)

In [146]:
from langchain.text_splitter import LatexTextSplitter

def split_latex_file(file_path, chunk_size=100, chunk_overlap=20):
    """
    Read LaTeX content from a file and split it into chunks.

    Args:
        file_path (str): Path to the LaTeX file
        chunk_size (int): Maximum size of each chunk
        chunk_overlap (int): Overlap between chunks

    Returns:
        list: List of LaTeX chunks
    """
    # Read the LaTeX content from the file
    try:
        with open(file_path, 'r', encoding='utf-8') as file:
            latex_text = file.read()
    except FileNotFoundError:
        print(f"Error: File '{file_path}' not found.")
        return []
    except Exception as e:
        print(f"Error reading file: {e}")
        return []

    # Create a splitter with the specified parameters
    splitter = LatexTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap
    )

    # Split the text into chunks
    chunks = splitter.split_text(latex_text)
    return chunks

In [147]:
chunks = split_latex_file(OUTPUT_FILE_PATH)

### Merging the additional data from Latex chunks with original data from merged_transcript.txt

In [148]:
# Convert latex chunks to Document objects
latex_documents = []
for i, chunk in enumerate(chunks):
    # Create Document object with page_content and metadata
    doc = Document(
        page_content=chunk,
        metadata={"source": f"{OUTPUT_FILE_PATH}", "chunk_id": i}
    )
    latex_documents.append(doc)

# Add the new LaTeX document chunks to the existing vector store
vector_store.add_documents(documents=latex_documents)

['e7f5f55f-eed3-4874-a24e-444940cc1e1d',
 '31c6e282-b5ca-4958-af46-8c6ccfce434f',
 'd39ade38-07ba-4d2b-ae71-3112a85c5ff5',
 'e6c4d57b-7e7a-457f-8e19-8a8a6e4deeec',
 'eb51f44e-6ed9-4a51-b182-815892a8dfd6',
 '458f9ab4-46eb-48e2-9817-c6c4b2b3d141',
 'a8f64fed-a8dc-4ad4-8caf-e116a047ae81',
 'd351bb16-389c-416b-9764-5a785b28365c',
 '7e580a81-31a5-4965-9c0d-a897662a5148',
 '1e4ee646-c6cc-43d0-8d13-df4799498092',
 '54ce10ca-5dde-41d4-89d8-86ca06b3fe04',
 '1d6929db-a8ac-4883-b26a-ebadd3d953c3',
 'dddea146-990f-48f5-bfbc-1d55805a5183',
 '38201bb2-0ce6-4818-ba40-54a5fca68931',
 '9279551d-96f1-49cb-8bf5-d30264a7f8a5',
 '62fa3e9e-733a-462f-8d83-fa27bdeeeca0',
 'c47ff0f7-1752-4783-8e2f-67d73bbdce4f',
 'e70f4b33-d7f6-416c-a0c9-5ae473afbcd6',
 '52613799-f9c6-4eac-a801-29f749e8ee42',
 '6023935c-adaa-4704-9a38-efb32df85a07',
 '14ac4729-8261-4224-87dc-5c93dbafbad1',
 'a763d249-4423-4fbb-a421-6bd144a7ee96',
 '12849682-a85d-4250-89d8-2f34fb4e3c76',
 '334cd6ac-51b0-4b75-9e3e-427d0a058c0f',
 '9d2846ae-ad9e-

In [149]:
# Test that both document sets are being used
TEST_QUERY = 'A matrix norm $\|\cdot\| : \mathbb{R}^{m \times n} \rightarrow \mathbb{R}$ satisfies:'

test_results = vector_store.similarity_search(TEST_QUERY, k=4)
for i, doc in enumerate(test_results):
    print(f"Result {i+1} source: {doc.metadata['source']}")
    print(f"Content preview: {doc.page_content[:100]}...\n")

Result 1 source: /content/drive/MyDrive/RAG_data/note_merged.tex
Content preview: $.

\subsection*{Vector Norms}

A function $\|\cdot\| : \mathbb{R}^d \rightarrow \mathbb{R}...

Result 2 source: /content/drive/MyDrive/RAG_data/note_merged.tex
Content preview: $.

\subsection*{Vector Norms}

A function $\|\cdot\| : \mathbb{R}^d \rightarrow \mathbb{R}...

Result 3 source: /content/drive/MyDrive/RAG_data/note_merged.tex
Content preview: $.

\subsection*{Vector Norms}

A function $\|\cdot\| : \mathbb{R}^d \rightarrow \mathbb{R}...

Result 4 source: /content/drive/MyDrive/RAG_data/note_merged.tex
Content preview: $.

\subsection*{Vector Norms}

A function $\|\cdot\| : \mathbb{R}^d \rightarrow \mathbb{R}...



## Leveraging tool-calling

source: https://python.langchain.com/docs/tutorials/qa_chat_history/

Leveraging tool-calling to interact with a retrieval step has another benefit, which is that the query for the retrieval is generated by our model. This is especially important in a conversational setting, where user queries may require contextualization based on the chat history. For instance, consider the following exchange:

Human: "What is Task Decomposition?"

AI: "Task decomposition involves breaking down complex tasks into smaller and simpler steps to make them more manageable for an agent or model."

Human: "What are common ways of doing it?"

In this scenario, a model could generate a query such as "common approaches to task decomposition". Tool-calling facilitates this naturally. As in the query analysis section of the RAG tutorial, this allows a model to rewrite user queries into more effective search queries. It also provides support for direct responses that do not involve a retrieval step (e.g., in response to a generic greeting from the user).

In [182]:
from langgraph.graph import MessagesState, StateGraph

graph_builder = StateGraph(MessagesState)

In [183]:
from langchain_core.tools import tool


@tool(response_format="content_and_artifact")
def retrieve(query: str):
    """Retrieve information related to a query."""
    retrieved_docs = vector_store.similarity_search(query, k=2)
    serialized = "\n\n".join(
        (f"Source: {doc.metadata}\n" f"Content: {doc.page_content}")
        for doc in retrieved_docs
    )
    return serialized, retrieved_docs

In [184]:
from langchain_core.messages import SystemMessage
from langgraph.prebuilt import ToolNode


# Step 1: Generate an AIMessage that may include a tool-call to be sent.
def query_or_respond(state: MessagesState):
    """Generate tool call for retrieval or respond."""
    llm_with_tools = llm.bind_tools([retrieve])
    response = llm_with_tools.invoke(state["messages"])
    # MessagesState appends messages to state instead of overwriting
    return {"messages": [response]}


# Step 2: Execute the retrieval.
tools = ToolNode([retrieve])


# Step 3: Generate a response using the retrieved content.
def generate(state: MessagesState):
    """Generate answer."""
    # Get generated ToolMessages
    recent_tool_messages = []
    for message in reversed(state["messages"]):
        if message.type == "tool":
            recent_tool_messages.append(message)
        else:
            break
    tool_messages = recent_tool_messages[::-1]

    # Format into prompt
    docs_content = "\n\n".join(doc.content for doc in tool_messages)
    system_message_content = (
        "You are an assistant for question-answering tasks. "
        "Use the following pieces of retrieved context to answer "
        "the question. If you don't know the answer, say that you "
        "don't know. Use three sentences maximum and keep the "
        "answer concise."
        "\n\n"
        f"{docs_content}"
    )
    conversation_messages = [
        message
        for message in state["messages"]
        if message.type in ("human", "system")
        or (message.type == "ai" and not message.tool_calls)
    ]
    prompt = [SystemMessage(system_message_content)] + conversation_messages

    # Run
    response = llm.invoke(prompt)
    return {"messages": [response]}

In [185]:
from langgraph.graph import END
from langgraph.prebuilt import ToolNode, tools_condition

graph_builder.add_node(query_or_respond)
graph_builder.add_node(tools)
graph_builder.add_node(generate)

graph_builder.set_entry_point("query_or_respond")
graph_builder.add_conditional_edges(
    "query_or_respond",
    tools_condition,
    {END: END, "tools": "tools"},
)
graph_builder.add_edge("tools", "generate")
graph_builder.add_edge("generate", END)

graph = graph_builder.compile()

In [186]:
input_message = "What is Task Decomposition?"

for step in graph.stream(
    {"messages": [{"role": "user", "content": input_message}]},
    stream_mode="values",
):
    step["messages"][-1].pretty_print()

================================ Human Message =================================

What is Task Decomposition?
================================== Ai Message ==================================
Tool Calls:
  retrieve (b0b8bd74-2da1-4e9c-836f-4be610f3aff0)
 Call ID: b0b8bd74-2da1-4e9c-836f-4be610f3aff0
  Args:
    query: Task Decomposition
================================= Tool Message =================================
Name: retrieve

Source: {'chunk_id': 395, 'source': '/content/drive/MyDrive/RAG_data/note_merged.tex'}
Content: $x_i \in \mathbb{R}^d$, the task is to partition the data into $k

Source: {'chunk_id': 395, 'source': '/content/drive/MyDrive/RAG_data/note_merged.tex'}
Content: $x_i \in \mathbb{R}^d$, the task is to partition the data into $k
================================== Ai Message ==================================

The task is to partition the data into k. The data is represented as $x_i \in \mathbb{R}^d$.


In [187]:
input_message = "How does Backpropagation work?"

for step in graph.stream(
    {"messages": [{"role": "user", "content": input_message}]},
    stream_mode="values",
):
    step["messages"][-1].pretty_print()

================================ Human Message =================================

How does Backpropagation work?
================================== Ai Message ==================================
Tool Calls:
  retrieve (19d50180-1a5c-4cff-9a6c-4cc93e0f8381)
 Call ID: 19d50180-1a5c-4cff-9a6c-4cc93e0f8381
  Args:
    query: How does Backpropagation work?
================================= Tool Message =================================
Name: retrieve

Source: {'source': '/content/drive/MyDrive/RAG_data/merged_transcript.txt'}
Content: 420
01:00:35.262 --> 01:00:36.492
Nilesh Gupta: So


421
01:00:37.476 --> 01:00:54.462
Nilesh Gupta: that's your chain rule, and like back provision typically involves like a forward pass and a backward pass forward pass is the like, the the what we already saw that, like in neural networks like how to compute the final output of your function. So and so forward pass just means, like you in


422
01:00:54.895 --> 01:01:16.952
Nilesh Gupta: along with computing 

In [188]:
input_message = "What are common ways of doing backpropagation?"

for step in graph.stream(
    {"messages": [{"role": "user", "content": input_message}]},
    stream_mode="values",
):
    step["messages"][-1].pretty_print()

================================ Human Message =================================

What are common ways of doing backpropagation?
================================== Ai Message ==================================
Tool Calls:
  retrieve (e79aa95c-d133-4243-9ab0-70c52a482fb6)
 Call ID: e79aa95c-d133-4243-9ab0-70c52a482fb6
  Args:
    query: common ways of performing backpropagation
================================= Tool Message =================================
Name: retrieve

Source: {'source': '/content/drive/MyDrive/RAG_data/merged_transcript.txt'}
Content: 420
01:00:35.262 --> 01:00:36.492
Nilesh Gupta: So


421
01:00:37.476 --> 01:00:54.462
Nilesh Gupta: that's your chain rule, and like back provision typically involves like a forward pass and a backward pass forward pass is the like, the the what we already saw that, like in neural networks like how to compute the final output of your function. So and so forward pass just means, like you in


422
01:00:54.895 --> 01:01:16.952
Nilesh G

### aspects to test

* Does context retrieval improve the answer quality? (Ask the same question but without the context material from RAG)
* Does tool-calling improve the quality of the answer?
* Test different chunk sizes


In [189]:
input_message = "What are common ways of doing backpropagation?"

for step in graph.stream(
    {"messages": [{"role": "user", "content": input_message}]},
    stream_mode="values",
):
    step["messages"][-1].pretty_print()

================================ Human Message =================================

What are common ways of doing backpropagation?
================================== Ai Message ==================================
Tool Calls:
  retrieve (e3cb11c3-a558-44d6-9996-68c1a776f030)
 Call ID: e3cb11c3-a558-44d6-9996-68c1a776f030
  Args:
    query: common ways of performing backpropagation
================================= Tool Message =================================
Name: retrieve

Source: {'source': '/content/drive/MyDrive/RAG_data/merged_transcript.txt'}
Content: 420
01:00:35.262 --> 01:00:36.492
Nilesh Gupta: So


421
01:00:37.476 --> 01:00:54.462
Nilesh Gupta: that's your chain rule, and like back provision typically involves like a forward pass and a backward pass forward pass is the like, the the what we already saw that, like in neural networks like how to compute the final output of your function. So and so forward pass just means, like you in


422
01:00:54.895 --> 01:01:16.952
Nilesh G

In [290]:
import pickle

eval_dataset_path = "/content/drive/MyDrive/generator_eval_dataset_gemini.pkl"
# Read the results back from the pickle file for testing
with open(eval_dataset_path, "rb") as f:
    loaded_results = pickle.load(f)

print(f"Total number of questions: {len(loaded_results)}")
sample = loaded_results[30:33]
for result in sample:
    print(result)
    print("-" * 35 + "\n")

Total number of questions: 77
{'question': 'According to the lecture, what is a key property of the K-truncated SVD of a matrix A, and what does Uk represent in the K-truncated SVD A ≈ Uk Σk Vkᵀ?', 'answer': 'A key property of the K-truncated SVD is that it provides the best rank-k approximation to the given matrix A. In the expression A ≈ Uk Σk Vkᵀ, Uk belongs to R^(M x K) and contains the first K singular vectors corresponding to the left singular vectors of A.'}
-----------------------------------

{'question': 'In the context of polynomial fitting using linear regression, what is the impact of increasing the polynomial degree (M) on the fit, and why might a high degree not be ideal for prediction?', 'answer': 'Increasing the polynomial degree (M) allows the model to fit the training data more closely, potentially achieving zero error on the training set. However, a high degree can lead to overfitting, where the model fits the noise in the training data rather than the underlying re

In [291]:
from langchain.schema import AIMessage, HumanMessage

results = []
for s in sample:
    for step in graph.stream(
        {"messages": [{"role": "user", "content": s['question']}]},
        stream_mode="values",
    ):
        out = step['messages'][-1]
        if isinstance(out, HumanMessage):
            print(f"Human: {out.content}")
        elif isinstance(out, AIMessage) and out.content:
            print(f"AI: {out.content}")
            results.append({'ai': out.content, 'ground_truth': s['answer']})
        elif out.content.startswith("Source:"):
            source = out.content.split("'source': '")[1].split("'")[0]
    print(f"Source: {source}\n" + "-"*50)

Human: According to the lecture, what is a key property of the K-truncated SVD of a matrix A, and what does Uk represent in the K-truncated SVD A ≈ Uk Σk Vkᵀ?
AI: A key property of the K-truncated SVD of a matrix A is that you can take any K. The truncated SVD is written as A ≈ Uk Σk Vkᵀ. In this expression, Uk is an R(m x k) matrix and is orthogonal.
Source: /content/drive/MyDrive/RAG_data/merged_transcript.txt
--------------------------------------------------
Human: In the context of polynomial fitting using linear regression, what is the impact of increasing the polynomial degree (M) on the fit, and why might a high degree not be ideal for prediction?
AI: Increasing the polynomial degree to fit training data exactly can lead to a poor job if a new test point came up. This phenomenon, known as overfitting, is characterized by many very large coefficients. Therefore, a high degree might not be ideal for prediction.

Source: /content/drive/MyDrive/RAG_data/merged_transcript.txt
------

In [292]:
from sklearn.metrics.pairwise import cosine_similarity
for r in results:
  ai_answer = r['ai']
  ground_truth = r['ground_truth']
  vec_ai = embeddings.embed_documents([ai_answer])
  vec_gt = embeddings.embed_documents([ground_truth])
  print(f"Similarity Score: {cosine_similarity(vec_ai, vec_gt)[0][0]:.4f}")

Similarity Score: 0.9233
Similarity Score: 0.9282
Similarity Score: 0.9037
